<a href="https://colab.research.google.com/github/juanperez20/javascript/blob/main/Codigo_copetencia_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q lightgbm xgboost catboost

import os
import shutil
import time
import warnings
import numpy as np
import pandas as pd
from google.colab import drive, files

warnings.filterwarnings("ignore")

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

inicio_total = time.time()

# ==============================================================================
# 1. CONEXIÓN PERMANENTE A GOOGLE DRIVE Y CARGA DEL ARCHIVO
# ==============================================================================
drive.mount("/content/drive", force_remount=False)

carpeta_proyecto = "/content/drive/MyDrive/Kaggle_California_UD"
os.makedirs(carpeta_proyecto, exist_ok=True)

nombre_fijo = "sample_submission.csv"
ruta_permanente = os.path.join(carpeta_proyecto, nombre_fijo)
reemplazar = False  # Cambiar a True solo si deseas reemplazar el archivo guardado

if not os.path.exists(ruta_permanente) or reemplazar:
    print("Selecciona el archivo para guardar/reemplazar en Google Drive:")
    subidos = files.upload()
    archivo_subido = list(subidos.keys())[0]
    if os.path.exists(ruta_permanente):
        os.remove(ruta_permanente)
    shutil.move(archivo_subido, ruta_permanente)

Archivo = ruta_permanente

def leer_csv_flexible(nombre_archivo):
    df_tmp = pd.read_csv(nombre_archivo)
    if df_tmp.shape[1] == 1:
        col_unica = df_tmp.columns[0]
        cols = [c.strip().strip('"') for c in col_unica.split(",")]
        df_tmp = df_tmp[col_unica].astype(str).str.strip('"').str.split(",", expand=True)
        df_tmp.columns = cols
        for c in df_tmp.columns:
            if c != "ocean_proximity":
                df_tmp[c] = pd.to_numeric(df_tmp[c], errors="coerce")
    return df_tmp

df_archivo = leer_csv_flexible(Archivo)

ruta_train_drive = os.path.join(carpeta_proyecto, "train.csv")
ruta_test_drive = os.path.join(carpeta_proyecto, "test.csv")

if os.path.exists(ruta_train_drive) and os.path.exists(ruta_test_drive):
    train = leer_csv_flexible(ruta_train_drive)
    test = leer_csv_flexible(ruta_test_drive)
elif os.path.exists("train.csv") and os.path.exists("test.csv"):
    train = leer_csv_flexible("train.csv")
    test = leer_csv_flexible("test.csv")
elif set(df_archivo.columns) == {"Id", "median_house_value"}:
    url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
    full_df = pd.read_csv(url)
    full_df["Id"] = full_df.index

    test_ids = df_archivo["Id"].astype(int)
    train = full_df[~full_df["Id"].isin(test_ids)].copy()
    test = full_df.set_index("Id").loc[test_ids].reset_index()
else:
    if "Id" not in df_archivo.columns:
        df_archivo = df_archivo.reset_index().rename(columns={"index": "Id"})
    train = df_archivo[df_archivo["median_house_value"].notna()].copy()
    test = df_archivo[df_archivo["median_house_value"].isna()].copy()

train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

X_train_raw = train.drop(columns=["Id", "median_house_value"], errors="ignore")
y_train = train["median_house_value"].values
X_test_raw = test.drop(columns=["Id", "median_house_value"], errors="ignore")

# ==============================================================================
# 2. INGENIERÍA DE CARACTERÍSTICAS ESPACIAL PROFUNDA (OUT-OF-FOLD KNN + GEODESIA)
# ==============================================================================
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

class UltraCaliforniaFeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=40, gamma=1.2, k_neighbors_list=(1, 2, 3, 5, 10, 20, 40)):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.k_neighbors_list = k_neighbors_list
        self.ciudades = {
            "sf": (37.7749, -122.4194),
            "sj": (37.3382, -121.8863),
            "oak": (37.8044, -122.2712),
            "la": (34.0522, -118.2437),
            "sm": (34.0195, -118.4912),   # Santa Monica (alto valor costero)
            "nb": (33.6189, -117.9298),   # Newport Beach
            "sd": (32.7157, -117.1611),
            "sb": (34.4208, -119.6982),   # Santa Barbara
            "sac": (38.5816, -121.4944),
            "fresno": (36.7378, -119.7871)
        }

    def _base_features(self, X):
        df = X.copy()
        # Imputación rápida interna para ratios
        bedrooms = df["total_bedrooms"].fillna(df["total_bedrooms"].median())
        rooms = df["total_rooms"].fillna(df["total_rooms"].median())
        households = df["households"].fillna(df["households"].median())
        pop = df["population"].fillna(df["population"].median())
        inc = df["median_income"].fillna(df["median_income"].median())
        age = df["housing_median_age"].fillna(df["housing_median_age"].median())

        # Ratios estructurales y socioeconómicos
        df["rooms_per_household"] = rooms / (households + 1e-5)
        df["bedrooms_per_room"] = bedrooms / (rooms + 1e-5)
        df["bedrooms_per_household"] = bedrooms / (households + 1e-5)
        df["population_per_household"] = pop / (households + 1e-5)
        df["population_per_room"] = pop / (rooms + 1e-5)
        df["income_per_room"] = inc / (df["rooms_per_household" ] + 1e-5)
        df["income_per_bedroom"] = inc / (df["bedrooms_per_household"] + 1e-5)
        df["income_per_person"] = inc / (df["population_per_household"] + 1e-5)
        df["income_sq"] = inc ** 2
        df["income_cube"] = inc ** 3
        df["income_x_age"] = inc * age
        df["is_age_capped"] = (age >= 52).astype(int)

        # Logaritmos
        for col in ["total_rooms", "total_bedrooms", "population", "households", "median_income"]:
            df[f"log_{col}"] = np.log1p(np.maximum(df[col].fillna(0), 0))

        # Distancias a polos urbanos y costeros
        dist_cols = []
        for nombre, (clat, clon) in self.ciudades.items():
            col_name = f"dist_{nombre}"
            df[col_name] = haversine_distance(df["latitude"], df["longitude"], clat, clon)
            df[f"inv_dist_{nombre}"] = 1.0 / (df[col_name] + 1.0)
            df[f"inc_over_dist_{nombre}"] = inc / (df[col_name] + 1.0)
            dist_cols.append(col_name)

        df["min_dist_city"] = df[dist_cols].min(axis=1)
        df["mean_dist_city"] = df[dist_cols].mean(axis=1)

        # Rotaciones espaciales multi-ángulo (15, 30, 45, 60 grados)
        for deg in [15, 30, 45, 60]:
            rad = np.radians(deg)
            df[f"rot_{deg}_x"] = df["longitude"] * np.cos(rad) - df["latitude"] * np.sin(rad)
            df[f"rot_{deg}_y"] = df["longitude"] * np.sin(rad) + df["latitude"] * np.cos(rad)

        # Coordenadas cartesianas 3D sobre la esfera terrestre
        lat_rad = np.radians(df["latitude"])
        lon_rad = np.radians(df["longitude"])
        df["geo_x"] = np.cos(lat_rad) * np.cos(lon_rad)
        df["geo_y"] = np.cos(lat_rad) * np.sin(lon_rad)
        df["geo_z"] = np.sin(lat_rad)

        return df

    def fit(self, X, y):
        df = self._base_features(X)
        self.y_train_ = np.asarray(y)

        # Clústeres espaciales KMeans
        self.kmeans_ = KMeans(n_clusters=self.n_clusters, random_state=42, n_init=10)
        self.kmeans_.fit(df[["latitude", "longitude"]])

        # Índice espacial en radianes (Haversine) para vecinos más cercanos exactos
        coords_rad = np.radians(df[["latitude", "longitude"]].values)
        max_k = max(self.k_neighbors_list) + 1
        self.nn_geo_ = NearestNeighbors(n_neighbors=max_k, metric="haversine", n_jobs=-1)
        self.nn_geo_.fit(coords_rad)

        # Índice combinado espacio + ingreso escalado
        self.scaler_sp_inc_ = RobustScaler()
        sp_inc = self.scaler_sp_inc_.fit_transform(
            np.column_stack([df["latitude"], df["longitude"], df["median_income"].fillna(df["median_income"].median()) * 1.5])
        )
        self.nn_sp_inc_ = NearestNeighbors(n_neighbors=max_k, metric="euclidean", n_jobs=-1)
        self.nn_sp_inc_.fit(sp_inc)

        self.train_coords_rad_ = coords_rad
        self.train_sp_inc_ = sp_inc
        self.train_income_ = df["median_income"].fillna(df["median_income"].median()).values
        return self

    def transform(self, X, is_train=False):
        df = self._base_features(X)

        # Similitud RBF a clústeres
        cluster_sims = rbf_kernel(df[["latitude", "longitude"]], self.kmeans_.cluster_centers_, gamma=self.gamma)
        for i in range(self.n_clusters):
            df[f"geo_sim_{i}"] = cluster_sims[:, i]

        # Vecinos espaciales puros (Haversine)
        coords_rad = np.radians(df[["latitude", "longitude"]].values)
        max_k = max(self.k_neighbors_list) + 1
        dists_geo, idxs_geo = self.nn_geo_.kneighbors(coords_rad, n_neighbors=max_k)

        # Si es el mismo conjunto de entrenamiento, excluimos el vecino 0 (el propio punto)
        start_col = 1 if is_train else 0
        end_col = max_k if is_train else max_k - 1

        dists_geo = dists_geo[:, start_col:end_col] * 6371.0  # en kilómetros
        idxs_geo = idxs_geo[:, start_col:end_col]

        df["nn_geo_dist_1"] = dists_geo[:, 0]
        df["nn_geo_dist_5_mean"] = dists_geo[:, :5].mean(axis=1)

        inc_actual = df["median_income"].fillna(np.median(self.train_income_)).values

        for k in self.k_neighbors_list:
            d_k = dists_geo[:, :k]
            i_k = idxs_geo[:, :k]
            y_k = self.y_train_[i_k]
            inc_k = self.train_income_[i_k]

            # Pesos por distancia inversa (IDW)
            w = 1.0 / (d_k + 0.05) ** 1.5
            w_norm = w / w.sum(axis=1, keepdims=True)

            df[f"knn_geo_idw_mean_{k}"] = (y_k * w_norm).sum(axis=1)
            df[f"knn_geo_mean_{k}"] = y_k.mean(axis=1)
            df[f"knn_geo_median_{k}"] = np.median(y_k, axis=1)
            df[f"knn_geo_std_{k}"] = y_k.std(axis=1)
            df[f"knn_geo_min_{k}"] = y_k.min(axis=1)
            df[f"knn_geo_max_{k}"] = y_k.max(axis=1)

            # Ajuste por diferencia de ingreso respecto al vecindario
            mean_inc_k = (inc_k * w_norm).sum(axis=1)
            df[f"knn_geo_inc_ratio_{k}"] = inc_actual / (mean_inc_k + 1e-4)
            df[f"knn_geo_adj_price_{k}"] = df[f"knn_geo_idw_mean_{k}"] * df[f"knn_geo_inc_ratio_{k}"]

        # Vecinos en espacio + ingreso conjunto
        sp_inc = self.scaler_sp_inc_.transform(
            np.column_stack([df["latitude"], df["longitude"], inc_actual * 1.5])
        )
        dists_si, idxs_si = self.nn_sp_inc_.kneighbors(sp_inc, n_neighbors=max_k)
        dists_si = dists_si[:, start_col:end_col]
        idxs_si = idxs_si[:, start_col:end_col]

        for k in [3, 8, 15]:
            d_k = dists_si[:, :k]
            i_k = idxs_si[:, :k]
            y_k = self.y_train_[i_k]
            w = 1.0 / (d_k + 0.02) ** 1.5
            w_norm = w / w.sum(axis=1, keepdims=True)
            df[f"knn_sp_inc_idw_{k}"] = (y_k * w_norm).sum(axis=1)

        return df

# ==============================================================================
# 3. VALIDACIÓN CRUZADA DE 10 PLIEGUES (10-FOLD) Y ENSAMBLE DE ALTA PRECISIÓN
# ==============================================================================
kf = KFold(n_splits=10, shuffle=True, random_state=42)

oof_lgbm = np.zeros(len(X_train_raw))
oof_xgb = np.zeros(len(X_train_raw))
oof_cat = np.zeros(len(X_train_raw))
oof_et = np.zeros(len(X_train_raw))

preds_lgbm = np.zeros(len(X_test_raw))
preds_xgb = np.zeros(len(X_test_raw))
preds_cat = np.zeros(len(X_test_raw))
preds_et = np.zeros(len(X_test_raw))

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["ocean_proximity"]),
    ],
    remainder=SimpleImputer(strategy="median")
)

print("Ejecutando validación cruzada de 10 pliegues con ingeniería espacial profunda...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_raw, y_train), 1):
    X_tr_raw, y_tr = X_train_raw.iloc[train_idx], y_train[train_idx]
    X_va_raw, y_va = X_train_raw.iloc[val_idx], y_train[val_idx]

    # Construcción de características sin fuga de datos (fit solo en train del pliegue)
    feat_builder = UltraCaliforniaFeatureBuilder(n_clusters=40, gamma=1.2)
    feat_builder.fit(X_tr_raw, y_tr)

    X_tr_df = feat_builder.transform(X_tr_raw, is_train=True)
    X_va_df = feat_builder.transform(X_va_raw, is_train=False)
    X_te_df = feat_builder.transform(X_test_raw, is_train=False)

    X_tr = preprocessor.fit_transform(X_tr_df)
    X_va = preprocessor.transform(X_va_df)
    X_te = preprocessor.transform(X_te_df)

    # 1. LightGBM
    m_lgbm = LGBMRegressor(
        n_estimators=2200,
        learning_rate=0.02,
        num_leaves=63,
        min_child_samples=15,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.75,
        reg_alpha=1.0,
        reg_lambda=5.0,
        random_state=100 + fold,
        verbose=-1,
        n_jobs=-1
    )
    m_lgbm.fit(X_tr, y_tr)
    oof_lgbm[val_idx] = m_lgbm.predict(X_va)
    preds_lgbm += m_lgbm.predict(X_te) / kf.n_splits

    # 2. XGBoost
    m_xgb = XGBRegressor(
        n_estimators=2200,
        learning_rate=0.02,
        max_depth=8,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.75,
        reg_alpha=1.0,
        reg_lambda=5.0,
        tree_method="hist",
        random_state=200 + fold,
        n_jobs=-1
    )
    m_xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx] = m_xgb.predict(X_va)
    preds_xgb += m_xgb.predict(X_te) / kf.n_splits

    # 3. CatBoost
    m_cat = CatBoostRegressor(
        iterations=2500,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=5,
        random_seed=300 + fold,
        verbose=0
    )
    m_cat.fit(X_tr, y_tr)
    oof_cat[val_idx] = m_cat.predict(X_va)
    preds_cat += m_cat.predict(X_te) / kf.n_splits

    # 4. ExtraTrees
    m_et = ExtraTreesRegressor(
        n_estimators=350,
        max_features=0.75,
        min_samples_leaf=2,
        random_state=400 + fold,
        n_jobs=-1
    )
    m_et.fit(X_tr, y_tr)
    oof_et[val_idx] = m_et.predict(X_va)
    preds_et += m_et.predict(X_te) / kf.n_splits

    rmse_fold = root_mean_squared_error(
        y_va,
        0.30 * oof_lgbm[val_idx] + 0.30 * oof_xgb[val_idx] + 0.25 * oof_cat[val_idx] + 0.15 * oof_et[val_idx]
    )
    print(f"Pliegue {fold:02d}/10 completado -> RMSE del pliegue: ${rmse_fold:,.2f}")

# ==============================================================================
# 4. META-OPTIMIZACIÓN DE PESOS (RIDGE STACKING) Y EXPORTACIÓN
# ==============================================================================
OOF_matrix = np.column_stack([oof_lgbm, oof_xgb, oof_cat, oof_et])
TEST_matrix = np.column_stack([preds_lgbm, preds_xgb, preds_cat, preds_et])

meta_learner = RidgeCV(alphas=np.logspace(-2, 5, 100))
meta_learner.fit(OOF_matrix, y_train)

pesos = meta_learner.coef_ / meta_learner.coef_.sum()
oof_final = OOF_matrix @ pesos
preds_finales = TEST_matrix @ pesos
preds_finales = np.clip(preds_finales, 14999, 500001)

# Guardar en Colab y en Google Drive (reemplazando siempre el anterior)
submission = pd.DataFrame({
    "Id": test["Id"].astype(int),
    "median_house_value": preds_finales
})

ruta_sub_drive = os.path.join(carpeta_proyecto, "submission.csv")
for ruta_out in ["submission.csv", ruta_sub_drive]:
    if os.path.exists(ruta_out):
        os.remove(ruta_out)
    submission.to_csv(ruta_out, index=False)

# ==============================================================================
# 5. REPORTE DE RESULTADOS
# ==============================================================================
rmse_lgbm = root_mean_squared_error(y_train, oof_lgbm)
rmse_xgb = root_mean_squared_error(y_train, oof_xgb)
rmse_cat = root_mean_squared_error(y_train, oof_cat)
rmse_et = root_mean_squared_error(y_train, oof_et)
rmse_final = root_mean_squared_error(y_train, oof_final)
mae_final = mean_absolute_error(y_train, oof_final)
r2_final = r2_score(y_train, oof_final)
tiempo_total = time.time() - inicio_total

print("\n" + "=" * 70)
print("RESULTADOS FINALES - MODELO DE ALTA PRECISIÓN (10-FOLD SPATIAL ENSEMBLE)")
print("=" * 70)
print(f"Tiempo total de ejecución: {tiempo_total / 60:.1f} minutos")
print(f"RMSE LightGBM:             ${rmse_lgbm:,.2f}")
print(f"RMSE XGBoost:              ${rmse_xgb:,.2f}")
print(f"RMSE CatBoost:             ${rmse_cat:,.2f}")
print(f"RMSE ExtraTrees:           ${rmse_et:,.2f}")
print("-" * 70)
print(f"Pesos óptimos aprendidos:  LGBM={pesos[0]:.3f}, XGB={pesos[1]:.3f}, CAT={pesos[2]:.3f}, ET={pesos[3]:.3f}")
print(f"RMSE FINAL DEL ENSAMBLE:   ${rmse_final:,.5f}")
print(f"MAE FINAL:                 ${mae_final:,.2f}")
print(f"R² FINAL:                  {r2_final:.5f} ({r2_final*100:.2f}% varianza explicada)")
print(f"Rango de predicciones:     ${preds_finales.min():,.2f} a ${preds_finales.max():,.2f}")
print(f"Mediana predicha vs real:  ${np.median(preds_finales):,.2f} vs ${np.median(y_train):,.2f}")
print(f"Archivo guardado en:       '{ruta_sub_drive}' y 'submission.csv'")
print("=" * 70)

files.download("submission.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ejecutando validación cruzada de 10 pliegues con ingeniería espacial profunda...
Pliegue 01/10 completado -> RMSE del pliegue: $39,934.07
Pliegue 02/10 completado -> RMSE del pliegue: $39,234.37
Pliegue 03/10 completado -> RMSE del pliegue: $40,368.30
Pliegue 04/10 completado -> RMSE del pliegue: $39,354.75
Pliegue 05/10 completado -> RMSE del pliegue: $41,265.29
Pliegue 06/10 completado -> RMSE del pliegue: $40,506.47
Pliegue 07/10 completado -> RMSE del pliegue: $42,998.44
Pliegue 08/10 completado -> RMSE del pliegue: $38,987.31
Pliegue 09/10 completado -> RMSE del pliegue: $41,416.28
Pliegue 10/10 completado -> RMSE del pliegue: $41,557.36

RESULTADOS FINALES - MODELO DE ALTA PRECISIÓN (10-FOLD SPATIAL ENSEMBLE)
Tiempo total de ejecución: 137.3 minutos
RMSE LightGBM:             $40,649.57
RMSE XGBoost:              $40,847.20
RMSE CatBoost:             $4

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>